In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Adaptez le chemin si votre CSV téléchargé depuis figshare porte un autre nom dans votre dossier
DATA_PATH = "data/Addis_Ababa_city_RTA.csv"
df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]

# Détection automatique de la colonne cible (nombre de victimes / casualties)
possible_targets = [
    c
    for c in df.columns
    if "casualt" in c.lower()
    and not any(
        bad in c.lower()
        for bad in ["class", "sever", "age", "sex", "fitness", "work"]
    )
]
TARGET_COL = possible_targets[0] if possible_targets else df.columns[-1]

print("=== PHASE 1.1 — STRUCTURE ET QUALITÉ ===")
print(f"Shape : {df.shape[0]} lignes x {df.shape[1]} colonnes")
print(f"Colonne cible détectée : '{TARGET_COL}'")
print("\nTypes de colonnes :\n", df.dtypes.value_counts())

print("\nValeurs manquantes (NaN) :")
na_counts = df.isna().sum()
print(na_counts[na_counts > 0] if na_counts.sum() > 0 else "0 NaN explicite")

# Valeurs 'unknown' / vides et catégories rares (< 1%)
cat_cols = df.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()
unk_dict, rare_dict = {}, {}
for col in cat_cols:
  s = df[col].astype(str).str.strip()
  n_unk = int(
      s.str.lower().isin(["unknown", "unk", "na", "n/a", "", "none", "nan"]).sum()
  )
  if n_unk > 0:
    unk_dict[col] = n_unk
  freq = s.value_counts(normalize=True)
  n_rare = int((freq < 0.01).sum())
  if n_rare > 0:
    rare_dict[col] = n_rare

print("\nColonnes avec 'unknown'/vides :", unk_dict)
print("Colonnes avec catégories rares (<1%) :", rare_dict)
print("Doublons exacts :", int(df.duplicated().sum()))

print("\n=== PHASE 1.2 — DISTRIBUTION DE LA CIBLE (G1) ===")
y_eda = pd.to_numeric(df[TARGET_COL], errors="coerce")
print(y_eda.describe())
print("\nRépartition des valeurs de la cible :")
print(y_eda.value_counts().sort_index())
print("\nSkewness brute :", round(float(y_eda.skew()), 4))
print("Skewness ln(1+y) :", round(float(np.log1p(y_eda.clip(lower=0)).skew()), 4))

# Graphique distribution cible brute vs ln(1+y)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(y_eda.dropna(), discrete=True, ax=axes[0], color="#4C72B0")
axes[0].set_title(f"Cible brute : {TARGET_COL}")
sns.histplot(
    np.log1p(y_eda.dropna().clip(lower=0)), ax=axes[1], color="#DD8452", kde=True
)
axes[1].set_title("Cible transformée : ln(1 + y)")
plt.tight_layout()
plt.show()

print("\n=== PHASE 1.3 — FUITE DE DONNÉES (LEAKAGE) ET BIAIS TH7 ===")
leakage_cols = [
    c
    for c in df.columns
    if c != TARGET_COL
    and any(k in c.lower() for k in ["casualty", "fitness", "severity"])
]
print("Colonnes à exclure (leakage) :", leakage_cols)
print(
    "Biais à documenter : biais de déclaration policière (reporting bias) sur "
    "les accidents relevés entre 2016 et 2022."
)